In [ ]:
import marimo as mo
import matplotlib.pyplot as plt
import cv2
import math

In [ ]:
img = cv2.imread('a5/img.jpg', cv2.IMREAD_GRAYSCALE)
h, w = img.shape
print(h,w)

In [ ]:
def output_img_frame(output_dim):
    output_img_frame = []
    for i in range(output_dim):
        row = []
        for j in range(output_dim):
            row.append(0)
        output_img_frame.append(row)
    return output_img_frame

In [ ]:
def bw_convert1(img,T=90):
    bw = img.copy()
    h, w = img.shape
    for i in range(h):
        for j in range(w):
            if img[i,j]>=T:
                bw[i,j] = 0
            else:
                bw[i,j] = 255
    return bw

bw_img=bw_convert1(img)
plt.imshow(bw_img, cmap='gray')
plt.show()

In [ ]:
def hough_transform(img,output_dim):
    diag = int(math.sqrt(output_dim**2+output_dim**2))
    diag = math.ceil(diag)
    rho_dim = int(2*diag+1) # zero and diag +ve and -ve
    theta_dim= 180 #0 to 180

    store_both =[]
    for i in range(rho_dim):
        rho=[]
        for j in range(theta_dim):
            rho.append(0)
        store_both.append(rho)

    ## calculate sin and cos vals first
    cos_vals=[]
    sin_vals=[]
    for a in range(theta_dim):
        deg = math.radians(a)
        cos_vals.append(math.cos(deg))
        sin_vals.append(math.sin(deg))

    ## detection
    for row in range(output_dim):
        for col in range(output_dim):
            if img[row][col] == 255:
                for a in range(theta_dim):
                    rho_val = col*cos_vals[a]+\
                            row*sin_vals[a]
                    rho_index = int(round(rho_val))+diag
                    store_both[rho_index][a] += 1
    return store_both, diag

def hough_detect(store_both, thresh, diag):
    lines=[]
    rho_rows = len(store_both)
    theta_cols = len(store_both[0]) # columns
    for r in range(rho_rows):
        for a in range(theta_cols):
            votes=store_both[r][a]
            if votes > thresh:
                detected_rho= r - diag
                detected_theta = a 
                lines.append((detected_rho, detected_theta))
    return lines

In [ ]:
store_both, diag = hough_transform(bw_img, h)
lines_detected = hough_detect(store_both, 210, diag)
print(f"the number of lines detected is {len(lines_detected)}")

In [ ]:
def draw_hough_lines(original_img, lines):
    output_viz = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for rho, theta in lines:
        a=math.cos(math.radians(theta))
        b=math.sin(math.radians(theta))
        x0=a*rho
        y0=b*rho
        # Extrapolate to draw the full line
        x1=int(x0+2000*(-b))
        y1=int(y0+2000*(a))
        x2=int(x0-2000*(-b))
        y2=int(y0-2000*(a))
        # Draw line in Red
        cv2.line(output_viz,(x1,y1),(x2,y2),(0,0,255), 2)     
    return output_viz
final_output = draw_hough_lines(hough_transform, lines_detected)

plt.figure(figsize=(10,10))
plt.imshow(final_output)
plt.show()

## Affine Transformations

In [ ]:
def rotation(img,angle):
    h, w=len(img), len(img[0])

    theta=math.radians(angle)
    cos_t=math.cos(theta)
    sin_t=math.sin(theta)
    new_w= int(abs(w*cos_t)+abs(h*sin_t))
    new_h= int(abs(w*sin_t)+abs(h*sin_t))

    rotated_img=[]
    for i in range(new_h):
        rotated_img.append([0]*new_w)
    cx,cy = w//2, h//2
    new_cx,new_cy = new_w//2, new_h//2

    for y in range(new_h):
        for x in range(new_w):
            x_shifted = x_new - new_cx
            y_shifted = y_new - new_cy            
            # Inverse Rotation Matrix (Rotate by -theta)
            # x_old = x_new * cos(t) + y_new * sin(t)
            # y_old = -x_new * sin(t) + y_new * cos(t)
            x_old = x_shifted * cos_t + y_shifted * sin_t
            y_old = -x_shifted * sin_t + y_shifted * cos_t

            # Shift back to original image coordinates
            x_src = int(round(x_old + cx))
            y_src = int(round(y_old + cy))
            #boundary
            if 0 <= x_src < w and 0 <= y_src < h:
                rotated_img[y_new][x_new] = img[y_src][x_src]

    return rotated_img

In [ ]:
def affine_skew(img, angle_deg):
    h, w = len(img), len(img[0])

    # Skew factor: tan(theta)
    tan_theta = math.tan(math.radians(angle_deg))

    # 1. Calculate New Dimensions
    # Height stays same, Width increases by h * tan(theta)
    skew_offset = int(abs(h * tan_theta))
    new_w = w + skew_offset
    new_h = h

    # Initialize Output Image
    skewed_img = []
    for i in range(new_h):
        skewed_img.append([0] * new_w)

    # Center offset to keep image in middle
    offset_x = 0
    if tan_theta < 0:
        offset_x = skew_offset # Shift right if skewing left

    # 2. Backward Mapping
    # Formula: x_new = x_old + y_old * tan(theta)
    # Inverse: x_old = x_new - y_new * tan(theta)
    for y_new in range(new_h):
        for x_new in range(new_w):

            y_src = y_new # Y doesn't change in horizontal skew

            # Calculate source X
            # We subtract the offset_x to handle negative skew angles correctly
            x_calc = (x_new - offset_x) - (y_src * tan_theta)
            x_src = int(round(x_calc))

            # 3. Boundary Check
            if 0 <= x_src < w and 0 <= y_src < h:
                skewed_img[y_new][x_new] = img[y_src][x_src]

    return skewed_img